# 🏫 Fine-Tuning GPT-2 on KIS Seoul Campus Dataset
### GPT-2 Fine-Tuning — KIS Seoul Q&A Chatbot

**Objective:** Fine-tune GPT-2 on the KIS Seoul Campus Q&A dataset so it can generate accurate, contextually relevant answers about the school.

**What you will learn:**
- How to prepare a custom Q&A dataset for language model training
- How to fine-tune GPT-2 using Hugging Face `transformers`
- How to generate text using the fine-tuned model

**References:** [Hugging Face Docs](https://huggingface.co/docs/transformers) | [GPT-2 Paper](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)

---
> ⚠️ **Before you start:** Go to **Runtime → Change runtime type → T4 GPU** to enable GPU acceleration.

## Step 1 — Verify GPU

In [1]:
!nvidia-smi

Fri May  1 17:31:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Step 2 — Install Dependencies

In [2]:
# Install required libraries
!pip install -q transformers datasets accelerate
print("Libraries installed successfully!")

Libraries installed successfully!


## Step 3 — Upload the Dataset

We use a `.txt` file where each Q&A pair is wrapped with special tokens:
```
<|startoftext|>
Question: ...
Answer: ...
<|endoftext|>
```
This teaches GPT-2 the **Question → Answer** pattern for KIS Seoul.

> 💡 **Tip:** Upload `kis_seoul_dataset.txt` using the Colab file panel (folder icon on the left), or mount your Google Drive below.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# ─── CHANGE THIS if your file is in a different location ───────────────────────
DATASET_FILE = "kis_seoul_dataset.txt"   # or full Drive path e.g. /content/drive/MyDrive/kis_seoul_dataset.txt
# ────────────────────────────────────────────────────────────────────────────────

import os
assert os.path.exists(DATASET_FILE), f"File not found: {DATASET_FILE}. Upload it or fix the path."
print(f"Dataset file found: {DATASET_FILE}")

# Preview first few lines
with open(DATASET_FILE, "r") as f:
    preview = f.read(600)
print("\n--- PREVIEW ---")
print(preview)

AssertionError: File not found: kis_seoul_dataset.txt. Upload it or fix the path.

## Step 4 — Load GPT-2 Model & Tokenizer

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "gpt2"   # Options: "gpt2", "gpt2-medium", "EleutherAI/gpt-neo-125M"

print(f"Loading model: {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# GPT-2 doesn't have a pad token by default — use EOS token as pad
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Model loaded!  Total parameters: {total_params:.1f}M")

## Step 5 — Tokenize and Prepare the Dataset

We convert raw text to token IDs and group them into fixed-size blocks of 128 tokens — the standard approach for causal language model fine-tuning.

In [ ]:
from datasets import load_dataset

# Load the text dataset
dataset = load_dataset("text", data_files=DATASET_FILE, split="train")
# Filter empty lines
dataset = dataset.filter(lambda x: len(x["text"].strip()) > 0)
print(f"Dataset loaded: {len(dataset)} lines")

# Tokenize
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=256)

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

# Group tokens into fixed-size blocks for language modeling
BLOCK_SIZE = 128

def group_texts(examples):
    concatenated = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = (len(concatenated["input_ids"]) // BLOCK_SIZE) * BLOCK_SIZE
    result = {
        k: [t[i : i + BLOCK_SIZE] for i in range(0, total_length, BLOCK_SIZE)]
        for k, t in concatenated.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

lm_dataset = tokenized_dataset.map(group_texts, batched=True)
print(f"Training blocks prepared: {len(lm_dataset)}")
print(f"Each block is {BLOCK_SIZE} tokens long")

Generating train split: 0 examples [00:00, ? examples/s]

Filter:   0%|          | 0/239 [00:00<?, ? examples/s]

Dataset loaded: 192 lines


Map:   0%|          | 0/192 [00:00<?, ? examples/s]

Map:   0%|          | 0/192 [00:00<?, ? examples/s]

Training blocks prepared: 24
Each block is 128 tokens long


## Step 6 — Configure Training

Key parameters to know:
- **`num_train_epochs`** — how many full passes over the data (more = better memorisation, but risk of overfitting)
- **`learning_rate`** — how fast the model updates weights
- **`fp16`** — mixed precision for faster training on GPU

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

OUTPUT_DIR = "./kis_seoul_gpt2_model"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=100,         # High epoch count for small dataset — forces memorisation
    learning_rate=1e-4,           # Slightly higher to speed up convergence
    per_device_train_batch_size=2,
    weight_decay=0.01,
    warmup_steps=50,
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,           # Keep only last 2 checkpoints to save disk
    prediction_loss_only=True,
    fp16=True,                    # Mixed precision — faster on T4/V100 GPU
    report_to="none",             # Disable Weights & Biases logging
)

# Data collator handles dynamic padding and MLM masking (mlm=False → causal LM)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_dataset,
    data_collator=data_collator,
)

print("Trainer configured successfully!")
print(f"Total training samples : {len(lm_dataset)}")
print(f"Epochs                 : {training_args.num_train_epochs}")
print(f"Batch size             : {training_args.per_device_train_batch_size}")

Trainer configured successfully!
Total training samples : 24
Epochs                 : 100
Batch size             : 2


## Step 7 — Train the Model

> ⏳ Training takes approximately **3–8 minutes** on a T4 GPU with this dataset size.  
> Watch the **loss** value — it should decrease over time. Lower loss = better model.

In [ ]:
print("Starting training...\n")
trainer.train()
print("\nTraining complete!")
# Show final loss
final_log = trainer.state.log_history[-1]
print(f"Final log: {final_log}")

Starting training...



Step,Training Loss
10,0.003898
20,0.020330
30,0.010521
40,0.014254
50,0.012351
60,0.015954
70,0.022108
80,0.044385
90,0.081606
100,0.048886


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training complete!
Final log: {'train_runtime': 207.9429, 'train_samples_per_second': 11.542, 'train_steps_per_second': 5.771, 'total_flos': 156775219200000.0, 'train_loss': 0.015590270613320172, 'epoch': 100.0, 'step': 1200}


## Step 8 — Save the Fine-Tuned Model

In [ ]:
SAVE_DIR = "./kis_seoul_gpt2_finetuned"

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print(f"Model saved to '{SAVE_DIR}'")
!ls -lh {SAVE_DIR}

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to './kis_seoul_gpt2_finetuned'
total 479M
-rw-r--r-- 1 root root  962 Apr 30 16:13 config.json
-rw-r--r-- 1 root root  118 Apr 30 16:13 generation_config.json
-rw-r--r-- 1 root root 475M Apr 30 16:13 model.safetensors
-rw-r--r-- 1 root root  297 Apr 30 16:13 tokenizer_config.json
-rw-r--r-- 1 root root 3.4M Apr 30 16:13 tokenizer.json


## Step 9 — Load the Fine-Tuned Model for Inference

In [ ]:
from transformers import pipeline

# Load the saved model into a text-generation pipeline
generator = pipeline(
    "text-generation",
    model=SAVE_DIR,
    tokenizer=SAVE_DIR,
    device=0   # 0 = GPU, -1 = CPU
)

print("Generator pipeline ready!")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Generator pipeline ready!


## Step 9 — Test the Model with KIS Seoul Prompts

Run a set of representative questions to evaluate how well the model learned the KIS Seoul dataset.

In [ ]:
prompts_to_test = [
    "Question: What is KIS Seoul Campus?\nAnswer:",
    "Question: What grades does KIS Seoul serve?\nAnswer:",
    "Question: How do I apply to KIS Seoul Campus?\nAnswer:",
    "Question: What languages are taught at KIS Seoul?\nAnswer:",
    "Question: Does KIS Seoul have a STEM program?\nAnswer:",
]

for prompt in prompts_to_test:
    print("=" * 60)
    print(f"PROMPT: {prompt.strip()}")
    outputs = generator(
        prompt,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.6,        # Lower = more focused / factual output
        top_k=40,               # Limits to top 40 token choices
        top_p=0.9,              # Nucleus sampling
        repetition_penalty=1.3,
        no_repeat_ngram_size=3,
        num_return_sequences=1,
        pad_token_id=tokenizer.eos_token_id,
    )
    generated = outputs[0]["generated_text"].replace(prompt, "").strip()
    print(f"GENERATED: {generated}")
    print()

Passing `generation_config` together with generation-related arguments=({'repetition_penalty', 'temperature', 'do_sample', 'no_repeat_ngram_size', 'pad_token_id', 'max_new_tokens', 'top_p', 'num_return_sequences', 'top_k'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT: Question: What is KIS Seoul Campus?
Answer:


Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


GENERATED: Korea International School (KIS) Seoul Campus is an international school located in Gangnam-gu, South Korean peninsula serving Pre-K to Grade 5, founded in 2000, offering an American-style curriculum with English as the medium of instruction. It is also a Council of International Schools (CISA), recognized by the Seoul Metropolitan Officeof Education, a member for EARCOS and other countries,

PROMPT: Question: What grades does KIS Seoul serve?
Answer:


Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


GENERATED: Korea International School (KIS) Seoul Campus serves students from Pre-K through Grade 5. It is an elementary-only campus serving Pre–K to Grade 5, founded in 2000, offering an American-style curriculum with English as the medium of instruction and an inquiry-based, transdisciplinary approach to teaching. The campus is committed to diversity and inclusion, welcoming students of all learning backgrounds and providing

PROMPT: Question: How do I apply to KIS Seoul Campus?
Answer:


Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


GENERATED: You canApply online through the OpenApply portal at kiskr.openapply.com

PROMPT: Question: What languages are taught at KIS Seoul?
Answer:


Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


GENERATED: Korean language is offered to all students. Chinese (Mandarin) is offered for students who require English Language support. Students in Kindergarten through Grade 5 can also enroll online with the Jeju Campus. Additional academic supports are also available depending on which criteria is met, including an admissions test and interview, a confidential letter of recommendation from the homeroom teacher, or both Yes & No answers.

PROMPT: Question: Does KIS Seoul have a STEM program?
Answer:
GENERATED: Yes. Kis Seoul has dedicated STEM specialist classes as part of both the Early Years and Elementary curriculum. The school is committed to applied learning with makerspaces and an inquiry-based, transdisciplinary approach to teaching. The School is small by design in order for students who require extensive instruction. The campus is nestled in park-like surroundings with trees, hiking trails, and river paths nearby. It



## Step 10 — Experiment & Reflect

Use the cell below to freely test your model with any prompt.

In [ ]:
my_prompt = "Question: What is KIS Seoul Campus?\nAnswer:"

outputs = generator(
    my_prompt,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.7,
    top_k=50,
    top_p=0.9,
    repetition_penalty=1.3,
    num_return_sequences=3,
    pad_token_id=tokenizer.eos_token_id,
)

print(f"Prompt: {my_prompt}\n")
for i, out in enumerate(outputs, 1):
    generated = out["generated_text"].replace(my_prompt, "").strip()
    print(f"[Output {i}]: {generated}\n")

Passing `generation_config` together with generation-related arguments=({'repetition_penalty', 'temperature', 'do_sample', 'pad_token_id', 'max_new_tokens', 'top_p', 'num_return_sequences', 'top_k'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prompt: Question: What is KIS Seoul Campus?
Answer:

[Output 1]: Korea International School (KIS) Seoul Campus is an international school located in Gangnam-gu, South Korean peninsula serving Pre-K to Grade 5, founded in 2000, offering an American-style curriculum with English as the medium of instruction. It is also a private, co-educational, non–sectarian national day school offering appropriate support for varied learners. The campus is committed to diversity and inclusion, welcoming students of all learning backgrounds and providing appropriate supports for varied learners throughout their academic lives

[Output 2]: Korea International School (KIS) Seoul Campus is an international school located in Gangnam-gu, South Korean peninsula serving Pre-K to Grade 5, founded in 2000, offering an American-style curriculum with English as the medium of instruction. It is fully accredited by WASC, the Western Association for Schools and Colleges. It is also a Council of International Schools 

---
## Summary

| Step | What happened |
|------|---------------|
| Dataset | KIS Seoul Q&A pairs formatted with GPT-2 special tokens (`<|startoftext|>` / `<|endoftext|>`) |
| Model | Loaded pre-trained GPT-2 (124M parameters) from Hugging Face |
| Tokenization | Converted text to token IDs in fixed-size blocks of 128 |
| Training | Fine-tuned GPT-2 on KIS Seoul data for 100 epochs |
| Inference | Generated contextual answers from a given prompt |

## 💡 Ideas to Extend This Project
- Add **more Q&A pairs** covering topics like fee structure, events, and staff
- Try **`gpt2-medium`** (345M params) for better generation quality (needs more VRAM)
- Try **`EleutherAI/gpt-neo-125M`** as an open-source alternative
- Build a simple **Gradio UI** so users can chat with the KIS Seoul bot
- Evaluate output quality using **BLEU score** or **perplexity**

---
